# NeuroSeg Optimized Training - 0.90+ Dice Target

This notebook runs the **Optimized Training Pipeline** (`train_optimized.py`) which includes:
1. **Attention U-Net** (with attention gates + dropout)
2. **Cosine Annealing Warm Restarts** (T0=30, Tmult=2)
3. **EMA Model Averaging** (smoother validation)
4. **Enhanced Augmentation** (9 transforms including elastic deformation)
5. **Optimized Loss** (0.5*Dice + 0.3*BCE + 0.2*Focal)

**Goal:** Reach **≥0.90 Dice Score** within **6 hours** on T4 GPU.

## Step 1: Connect to GPU
Go to **Runtime** → **Change runtime type** and select **T4 GPU**.

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Navigate to your project folder
PROJECT_PATH = '/content/drive/MyDrive/neuroseg'  # CHANGE THIS to your path
os.chdir(PROJECT_PATH)
print(f"Current Directory: {os.getcwd()}")

## Step 3: Install Dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless tqdm

## Step 6: Run Optimized Training

Running `train_optimized.py` (Attention U-Net + Cosine Annealing + EMA).

In [ ]:
!python3 train_optimized.py

## Step 6b: Live Metrics Monitor (Optional)

Run this cell **simultaneously** with training to see live updates!

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
import time
import threading
import os

def monitor_training_live():
    """Monitor training_history.json and plot metrics live"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('⏱️  LIVE Training Metrics - Optimized Attention U-Net', 
                 fontsize=14, fontweight='bold')
    
    history_file = 'training_history.json'
    
    while True:
        if os.path.exists(history_file):
            try:
                with open(history_file, 'r') as f:
                    history = json.load(f)
                
                # Metrics exist checking
                if not history['train_loss']:
                    time.sleep(2)
                    continue

                # Plot 1: Loss
                axes[0, 0].clear()
                axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2, alpha=0.7)
                axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
                axes[0, 0].set_title('Loss', fontweight='bold')
                axes[0, 0].set_xlabel('Epoch')
                axes[0, 0].legend()
                axes[0, 0].grid(True, alpha=0.3)
                
                # Plot 2: Dice
                axes[0, 1].clear()
                axes[0, 1].plot(history['train_dice'], label='Train', linewidth=2, alpha=0.7)
                axes[0, 1].plot(history['val_dice'], label='Val', linewidth=2, color='orange')
                max_dice = max(history['val_dice'])
                axes[0, 1].axhline(y=0.90, color='g', linestyle='--', alpha=0.5, label='Target (0.90)')
                axes[0, 1].set_title(f'Dice Coefficient (Max: {max_dice:.4f})', fontweight='bold')
                axes[0, 1].set_ylim([0, 1])
                axes[0, 1].legend()
                axes[0, 1].grid(True, alpha=0.3)
                
                # Plot 3: IoU
                axes[0, 2].clear()
                axes[0, 2].plot(history['train_iou'], label='Train', linewidth=2, alpha=0.7)
                axes[0, 2].plot(history['val_iou'], label='Val', linewidth=2, color='red')
                axes[0, 2].set_title('IoU Score', fontweight='bold')
                axes[0, 2].set_ylim([0, 1])
                axes[0, 2].legend()
                axes[0, 2].grid(True, alpha=0.3)
                
                # Plot 4: Smoothed Dice
                axes[1, 0].clear()
                val_dice = history['val_dice']
                window = min(10, len(val_dice) // 5)
                if window > 1:
                    smoothed = np.convolve(val_dice, np.ones(window)/window, mode='valid')
                    axes[1, 0].plot(val_dice, label='Raw', alpha=0.3, linewidth=1)
                    axes[1, 0].plot(range(window-1, len(val_dice)), smoothed, 
                                   label=f'Smoothed', linewidth=2, color='orange')
                else:
                    axes[1, 0].plot(val_dice, label='Dice', linewidth=2)
                axes[1, 0].set_title('Smoothness Check', fontweight='bold')
                axes[1, 0].grid(True, alpha=0.3)
                
                # Plot 5: Learning Rate
                axes[1, 1].clear()
                if 'learning_rates' in history and len(history['learning_rates']) > 0:
                    axes[1, 1].semilogy(history['learning_rates'], linewidth=2, color='purple')
                    axes[1, 1].set_title('LR Schedule (Log Scale)', fontweight='bold')
                    axes[1, 1].grid(True, alpha=0.3)
                
                # Plot 6: Overfitting Gap
                axes[1, 2].clear()
                gap = [t - v for t, v in zip(history['train_dice'], history['val_dice'])]
                axes[1, 2].plot(gap, linewidth=2, color='brown')
                axes[1, 2].axhline(y=0.05, color='g', linestyle='--', alpha=0.5, label='Good')
                axes[1, 2].set_title('Train-Val Gap', fontweight='bold')
                axes[1, 2].grid(True, alpha=0.3)
                
                plt.tight_layout()
                clear_output(wait=True)
                display(fig)
                
                epoch_num = len(history['val_dice'])
                current_dice = history['val_dice'][-1]
                print(f"📊 Epoch {epoch_num:3d} | Dice: {current_dice:.4f} | Max: {max_dice:.4f}")
                
            except Exception as e:
                pass
        
        time.sleep(5)

monitor_thread = threading.Thread(target=monitor_training_live, daemon=True)
monitor_thread.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nMonitor stopped")

## Step 7: Save & Download

Artifacts are automatically saved to Google Drive, but you can also download them here.

In [ ]:
from google.colab import files
if os.path.exists('best_model.pth'):
    files.download('best_model.pth')
if os.path.exists('training_history.json'):
    files.download('training_history.json')
if os.path.exists('training_metrics.png'):
    files.download('training_metrics.png')